In [1]:
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import os

from utils import GOOGLE_SHEETS, KNOWN_BRANDS

def get_gspread_client(json_path='../credentials.json'):
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"Nu găsesc fișierul de credențiale la: {json_path}")
        
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
    creds = ServiceAccountCredentials.from_json_keyfile_name(json_path, scope)
    return gspread.authorize(creds)

def load_sheet_by_gid(client, sheet_config, page_name, header_row=4):
    try:
        spreadsheet = client.open_by_key(sheet_config.id)
        gid_target = sheet_config.pages.get(page_name)
        if not gid_target:
            raise ValueError(f"Pagina '{page_name}' nu există în config pentru acest raport.")
            
        worksheet = None
        for ws in spreadsheet.worksheets():
            if str(ws.id) == str(gid_target):
                worksheet = ws
                break
        
        if not worksheet:
            raise ValueError(f"Nu am găsit worksheet cu GID: {gid_target}")

        print(f"Descarc datele din '{page_name}'...")
        all_values = worksheet.get_all_values()
        
        if len(all_values) <= header_row:
            raise ValueError("Fișierul pare gol sau header-ul e setat greșit.")
            
        headers = all_values[header_row]
        data_rows = all_values[header_row + 1:]
        
        df = pd.DataFrame(data_rows, columns=headers)
        return df

    except Exception as e:
        print(f"Eroare: {e}")
        return None

In [ ]:
client = get_gspread_client()

df_pacienti = load_sheet_by_gid(
    client, 
    GOOGLE_SHEETS['Raport_pacienti'], 
    'Pacienti',
    header_row=3 # Verifică în Excel dacă linia cu "Nume", "Prenume" e a 5-a (index 4)
)

if df_pacienti is not None:
    display(df_pacienti.head())
    print(f"Coloane detectate: {df_pacienti.columns.tolist()}")

Descarc datele din 'Pacienti'...


,CIRJA,ALINA,0,0745575388,cirja_ionela_alina@yahoo.com,2025-05-30 15:40:00,1
0,ABABEI,CATALIN,1931214045357,0737963664,lavinia.negrei@yahoo.com,2022-11-10 16:00:00,1
1,ABABEI,IOANA,2521003044421,0748818292,,2023-09-28 10:20:00,1
2,ABAGERU,GHEORGHE,1461205044435,0741738839,,2023-06-14 16:45:00,1
3,ABARBOAE,DARIA,,0748033207,,2023-02-08 10:15:00,1
4,ABURDULESEI,GEANINA,0,0734926721,,2025-04-10 09:20:00,1


Coloane detectate: [' CIRJA', 'ALINA', '0', '0745575388', 'cirja_ionela_alina@yahoo.com', '2025-05-30 15:40:00', '1']
